In [14]:
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report


x_train = pd.read_csv("modeling_data/x_train.csv")
x_test = pd.read_csv("modeling_data/x_test.csv")
y_train = pd.read_csv("modeling_data/y_train.csv").squeeze()
y_test = pd.read_csv("modeling_data/y_test.csv").squeeze()


print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(248, 5)
(62, 5)
(248,)
(62,)


بخش 1:

select modeling technique(انتخاب تکنیک مدل سازی)

با توجه به داده هام باید از clasification استفاده کنم 

چون فیلد هدف رو دارم 

فکر کنم استفاده ار logestic regression , decision tree برای این دیتاست منطقی تر باشه 

چون دادههای زیاد نداریم که از random forest استفاده کنیم 

هم نیاز به تفسیر داریم 

بخش 2:

generate test design (طراحی نحوه تست)

بخاطر اینکه دیتا ستم کلا 310 رکورد داره بنظرک باید از cross validatio استفاده کنم 

تا پایدار تر  قایل اعتماد تر بشه 

و چون فیلد هدف متوازن نیستش باید recall ,f1 score استفاده کنم

بخش 3:

build model(ساخت مدل )

In [15]:
scaler = StandardScaler()

x_train_scaled= scaler.fit_transform(x_train)
x_test_scaled= scaler.fit_transform(x_test)



In [16]:
Logistic_model = LogisticRegression()

Logistic_model.fit(x_train_scaled, y_train)
y_pred_logistic = Logistic_model.predict(x_test_scaled)

Logistic_model.coef_

array([[ 0.71374503, -0.2480162 , -0.71147408, -0.99593409,  3.20362342]])

با توجه به این اعداد بیشترین تاثیر رو degree_spondylolisthesis داره 

و کمترین تاثیر رو lumbar_lordosis_angle دازه 


In [17]:
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(x_train_scaled,y_train)
y_pred_tree = tree_model.predict(x_test_scaled)
tree_model.get_depth()

10

برای این تعداد رکورد واقعا عمق 10 زیاده و احتمالا مدل داره داده هارو حفظ میکنه و یاد نمیگیره

بخش 4:

assess model(ارزیابی اولیه مدل)

In [18]:
print("LogisticRegression: ", classification_report(y_test, y_pred_logistic), sep ="\n")
print("DecisionTree: ", classification_report(y_test, y_pred_tree), sep="\n")

LogisticRegression: 
              precision    recall  f1-score   support

           0       0.77      0.85      0.81        20
           1       0.93      0.88      0.90        42

    accuracy                           0.87        62
   macro avg       0.85      0.87      0.86        62
weighted avg       0.88      0.87      0.87        62

DecisionTree: 
              precision    recall  f1-score   support

           0       0.64      0.80      0.71        20
           1       0.89      0.79      0.84        42

    accuracy                           0.79        62
   macro avg       0.77      0.79      0.77        62
weighted avg       0.81      0.79      0.80        62



تو همه معیار ها logesticبهتر از decision tree عمل کرده 

 و چون دقت جفت مدل رو نرمال کمه بهتره cross validation انجام بدم



In [20]:
X = pd.concat([x_train, x_test], axis =0)
Y = pd.concat([y_train, y_test], axis=0)

x_scaled = StandardScaler().fit_transform(X)

log_cs_score = cross_val_score(LogisticRegression(), x_scaled , Y, cv=5,scoring='f1' )
tree_cs_score = cross_val_score(DecisionTreeClassifier(random_state=42), x_scaled ,Y , cv = 5, scoring='f1')

print("Logistic Regression f1 scores: ",log_cs_score)
print("Logistic Regression mean f1: ",log_cs_score.mean())

print("Decision tree f1 scores: ", tree_cs_score)
print("Decision tree mean f1: ", tree_cs_score.mean())


Logistic Regression f1 scores:  [0.88888889 0.88636364 0.86419753 0.9047619  0.91566265]
Logistic Regression mean f1:  0.8919749222962073
Decision tree f1 scores:  [0.86075949 0.86363636 0.825      0.79487179 0.85365854]
Decision tree mean f1:  0.8395852377528821


بااین ارزیابی بهم ثابت شد logistic واقعا بهتر از decision tree عمل میکنه چون 

دقت و f1 بهتری داره